In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from dotenv import load_dotenv
import urllib.parse
import os
import clickhouse_connect
import numpy as np
import pandas as pd

In [ ]:
orange = '#FF670F'
pink = '#FC006D'
blue = '#006CED'
green = '#2BD67C'
red = '#FF5D5D'
black = '#2B3034'

table='datawarehouse_prod.fact_payload' 
device="'6042702a83c2f34ea421ffbb'"
maxi = 10000000000000
sort1 = 'payload_timestamp'
sort2= 'payload_metric_id'
limit= (0,1000)

In [ ]:
load_dotenv()
host = urllib.parse.quote_plus(os.environ.get("clk_prod"))
username = urllib.parse.quote_plus(os.environ.get("USER_NAME"))
password = urllib.parse.quote_plus(os.environ.get('clk_prod_pass'))

client = clickhouse_connect.get_client(host=host, username=username, 
                                       port = 443, password=password)

# <span style="color:green"> 1 - Répartition sur les clients </span> 

### Nombre de clients ayant au moins un relevé

In [ ]:
df = client.query_df(f'SELECT count(temp.entity_id) FROM \
             (SELECT entity_id, count(payload_timestamp) nbPay\
             FROM {table}\
             GROUP BY entity_id\
             settings max_rows_to_read = {maxi}) AS temp\
             settings max_rows_to_read = {maxi}')
print(df)
# 480

### Distribution des payloads par client

In [ ]:
df = client.query_df(f'SELECT entity_id, count(payload_timestamp) nbPay\
             FROM {table}\
             GROUP BY entity_id\
             settings max_rows_to_read = {maxi}')

In [ ]:
bins = [0, 100]+df.describe().nbPay.tolist()[4:]
fig = go.Figure()
fig.add_trace(
    go.Histogram(x=pd.cut(df.nbPay, bins)\
        .sort_values().astype(str), \
                marker={'color': blue}))
fig.update_layout(title=f"Distribution des effectifs de relevés par client.",
                  xaxis_title="Nombre de relevés par tranche (quantile)", 
                  yaxis_title="Nombre de clients",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

# <span style="color:green"> 2 - Répartition sur les équipements </span> 

### Nombre d'équipements associés au moins à un relevé

In [ ]:
df = client.query_df(f'SELECT count(device_id) FROM \
             (SELECT device_id, count(payload_timestamp) nbPay\
             FROM {table}\
             GROUP BY device_id\
             settings max_rows_to_read = {maxi})\
             settings max_rows_to_read = {maxi}')
print(df)

### Distribution des payloads par équipement

In [ ]:
df = client.query_df(f'SELECT device_id, count(payload_timestamp) nbPay\
             FROM {table}\
             GROUP BY device_id\
             ORDER BY nbPay\
             settings max_rows_to_read = {maxi}')

In [ ]:
bins = [0, 100]+df.describe().nbPay.tolist()[4:]
fig = go.Figure()
fig.add_trace(
    go.Histogram(x=pd.cut(df.nbPay, bins)\
        .sort_values().astype(str), \
                marker={'color': blue}))
fig.update_layout(title=f"Distribution des effectifs de relevés par équipement.",
                  xaxis_title="Nombre de relevés par tranche (quantile)", 
                  yaxis_title="Nombre d'équipements",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

# <span style="color:green"> 3 - Répartition sur les capteurs </span> 

### Nombre de capteurs associés au moins à un relevé

In [ ]:
df = client.query_df(f'SELECT count(sens) FROM \
             (SELECT payload_metric_id sens, count(payload_timestamp) nbPay\
             FROM {table}\
             GROUP BY payload_metric_id\
             settings max_rows_to_read = {maxi})\
             settings max_rows_to_read = {maxi}')
print(df)

### Distribution des payloads par capteur

In [ ]:
df = client.query_df(f'SELECT payload_metric_id, count(payload_timestamp) nbPay\
             FROM {table}\
             GROUP BY payload_metric_id\
             ORDER BY nbPay\
             settings max_rows_to_read = {maxi}')

In [ ]:
bins = [0, 100]+df.describe().nbPay.tolist()[4:]
fig = go.Figure()
fig.add_trace(
    go.Histogram(x=pd.cut(df.nbPay, bins)\
        .sort_values().astype(str), \
                marker={'color': blue}))
fig.update_layout(title=f"Distribution des effectifs de relevés par capteur.",
                  xaxis_title="Nombre de relevés par tranche (quantile)", 
                  yaxis_title="Nombre de capteurs",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### Distribution des equipements par capteur

In [ ]:
df = client.query_df(f"""SELECT payload_metric_id, count(distinct device_id) nbDev\
             FROM {table}
             GROUP BY payload_metric_id\
             settings max_rows_to_read = {maxi}""")

In [ ]:
bins = [0,1,2,10,100,1000,10000,100000]
df['bins'] = pd.cut(df.nbDev, bins)
fig = go.Figure()
fig.add_trace(
    go.Histogram(x=df.bins.sort_values().astype(str), \
                marker={'color': blue}))
fig.update_yaxes(type='log')
fig.update_layout(title=f"Distribution des effectifs d'équipements associés aux capteurs",
                  xaxis_title="Nombre d'équipements par tranche (quantile)", 
                  yaxis_title="Nombre de capteurs <br> ",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

# <span style="color:green"> 3 - Couverture temporelle des données  </span> 

### Définition temporelle des relevés par équipement

In [ ]:
# SELECT t2.device_id, age('year', t2.begin, t2.end), t2.begin, t2.end
# FROM(
# 	SELECT device_id, MIN(payload_timestamp) begin, MAX(payload_timestamp) end
# 	FROM datawarehouse_prod.fact_payload
# 	GROUP BY device_id
# 	settings max_rows_to_read = 10000000000000) AS t2
# settings max_rows_to_read = 10000000000000;
df = pd.read_csv('../collection_payloads/data/device_id_time_coverage.csv')
df['fin'] = pd.to_datetime(df.fin)
df['debut'] = pd.to_datetime(df.debut)
df['delta']  = (pd.to_datetime(df.fin) - pd.to_datetime(df.debut))

In [ ]:
date_range = pd.date_range(start=df.debut.min(), end=df.fin.max(), freq="MS")
time_coverage_df = []
for ma_date in date_range:
    nbDev = df[(df.debut < ma_date) &  (df.fin > ma_date)].shape[0]
    time_coverage_df.append({
        'date':ma_date, "nbDev":nbDev})
time_coverage_df = pd.DataFrame(time_coverage_df)

In [ ]:
fig = go.Figure(go.Histogram(histfunc='sum', 
                             x=time_coverage_df.date, y=time_coverage_df.nbDev, 
                            marker={'color': green}))
# Set xaxis range
fig.update_xaxes(type='date', 
                 tickformat='%Y-%m-%d', 
                 nticks=10, 
                 range=[df.debut.min(), df.fin.max()])
fig.update_yaxes(type='log')
fig.update_layout(title=f"Distribution des équipements actifs sur le temps.",
                  xaxis_title="Date", 
                  yaxis_title="Nombre d'équipements actifs <br> (Echelle logarithmique)",
                  font_family="Arial", font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

### Définition temporelle des relevés par équipement

In [ ]:
# SELECT t2.device_id, age('year', t2.begin, t2.end), t2.begin, t2.end
# FROM(
# 	SELECT device_id, MIN(payload_timestamp) begin, MAX(payload_timestamp) end
# 	FROM datawarehouse_prod.fact_payload
# 	GROUP BY device_id
# 	settings max_rows_to_read = 10000000000000) AS t2
# settings max_rows_to_read = 10000000000000;
df = pd.read_csv('../collection_payloads/data/device_id_time_coverage.csv')
df['fin'] = pd.to_datetime(df.fin)
df['debut'] = pd.to_datetime(df.debut)
df['delta']  = (pd.to_datetime(df.fin) - pd.to_datetime(df.debut))

# <span style="color:green"> 4 - Répartition en fonction des sources </span> 

#### Nombre de payloads par type de source

In [ ]:
# df = client.query_df(f"""SELECT payload_source, count(payload_timestamp) nbPay\
#              FROM {table}\
#              GROUP BY payload_source\
#              settings max_rows_to_read = {maxi}""")
# df.to_csv('./data/source_nbPay.csv')
df = pd.read_csv('./data/source_nbPay.csv')

In [ ]:
df.sort_values('nbPay', inplace=True)
fig = go.Figure()
fig.add_trace(
    go.Bar(x = df.payload_source,
           y = df.nbPay,
        marker={'color': orange}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Répartition du volume par source de payload (nb < mediane)",
                  xaxis_title="Source", 
                  yaxis_title="Nombre de factures <br> (échelle logarithmique)",
                  font_family="Arial",
                font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

#### Nombre d'équipements par type de source

In [ ]:
df = client.query_df(f"""SELECT payload_source, uniqExact(device_id) AS nbDev\
                 FROM {table} \
            GROUP BY payload_source \
            settings max_rows_to_read = {maxi}""")

In [ ]:
df.sort_values('nbDev', inplace=True)
fig = go.Figure()
fig.add_trace(
    go.Bar(x = df.payload_source,
           y = df.nbDev,
        marker={'color': pink}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Répartition du volume d'équipements par source",
                  xaxis_title="Source", 
                  yaxis_title="Nombre d'équipements <br> (échelle logarithmique)",
                  font_family="Arial",
                font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1000, height=400,
                template="simple_white")
fig.show()

# <span style="color:green"> 5 - Répartition en fonction des familles d'équipement </span> 

#### Nombre de capteurs par famille d'équipement et par équipement

In [ ]:
df = client.query_df(f"""SELECT d.device_type, p.device_id, uniqExact(p.payload_metric_id) AS nbSens\
                 FROM {table} p\
                 JOIN (SELECT device_type, device_id\
                      FROM datawarehouse_prod.dim_device) d\
                      ON d.device_id = p.device_id\
            GROUP BY d.device_type, p.device_id\
            settings max_rows_to_read = {maxi}""")

# <span style="color:green"> 6 - Fréquence des données </span> 

Fréquence moyenne des relevés, en secondes par équipement

In [ ]:
# df = client.query_df(f"""SELECT device_id, arrayReduce('avg', diffs) avgDiff FROM( \
#     SELECT device_id, arrayDifference(arraySort(groupArray(payload_timestamp))) diffs\
#     FROM {table} \
#     GROUP BY device_id) \
# ORDER BY avgDiff DESC \
# settings max_rows_to_read = {maxi}""")
df = pd.read_csv('../collection_payloads/data/device_id_freq_moyenne.csv')

In [ ]:
df['hours'] = df.avgDiff/60/60
df['days'] = df.avgDiff/60/60/24
df['timedelta'] = df.avgDiff.apply(lambda x: pd.Timedelta(x, unit='s'))
fig = go.Figure()
fig.add_trace(
    go.Histogram(x = df[(df.timedelta > pd.Timedelta(24, unit='d'))].days,
        marker={'color': green}
    )
)
fig.update_xaxes(type="log")
fig.update_layout(title=f"Distribution des fréquences relatives d'émission des relevés par équipement (Supérieur à une journée)",
                  xaxis_title="Différence moyenne entre deux relevés (en jours).", 
                  yaxis_title="Nombre d'équipements",
                  font_family="Arial",
                font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1100, height=400,
                template="simple_white")
fig.show()

In [ ]:
df[df.timedelta > pd.Timedelta(24, unit='d')].timedelta.mean()
df[df.timedelta > pd.Timedelta(24, unit='d')].timedelta.describe()

Parmi les équipements dont la fréquence moyenne de relevé est supérieure à une journée: on compte ***3122*** instances.
En moyenne, la fréquence des relevés est ***mensuelle***.

In [ ]:
df['mn'] = df.avgDiff/60
fig = go.Figure()
fig.add_trace(
    go.Histogram(x = df[(df.timedelta < pd.Timedelta(24, unit='h'))& (df.timedelta > pd.Timedelta(3600, unit='s'))].mn,
        marker={'color': green}
    )
)
# fig.update_xaxes(type="log")
fig.update_layout(title=f"Distribution des fréquences relatives d'émission des relevés par équipement (entre 1h et 1jourée).",
                  xaxis_title="Différence moyenne entre deux relevés(en heures)",
                  yaxis_title="Nombre d'équipements",
                  font_family="Arial",
                font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1100, height=400,
                template="simple_white")
fig.show()

In [ ]:
df['mn'] = df.avgDiff/60
fig = go.Figure()
fig.add_trace(
    go.Histogram(x = df[(df.timedelta < pd.Timedelta(3600, unit='s'))& (df.timedelta > pd.Timedelta(60, unit='s'))].mn,
        marker={'color': green}
    )
)
# fig.update_xaxes(type="log")
fig.update_layout(title=f"Distribution des fréquences relatives d'émission des relevés par équipement (entre 1mn et 1h).",
                  xaxis_title="Différence moyenne entre deux relevés", 
                  yaxis_title="Nombre d'équipements",
                  font_family="Arial",
                font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1100, height=400,
                template="simple_white")
fig.show()

In [ ]:
df[(df.timedelta < pd.Timedelta(3600, unit='s'))& (df.timedelta > pd.Timedelta(60, unit='s'))].timedelta.mean()
df[(df.timedelta < pd.Timedelta(3600, unit='s'))& (df.timedelta > pd.Timedelta(60, unit='s'))].timedelta.describe()

Parmi les équipements dont la fréquence moyenne de relevé est comprise entre 1mn et 1h: on compte ***81k*** instances.
En moyenne, la fréquence des relevés est de ***23mn***.

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Histogram(x = df[(df.timedelta < pd.Timedelta(60, unit='s'))].avgDiff,
        marker={'color': green}
    )
)
# fig.update_xaxes(type="log")
fig.update_layout(title=f"Distribution des fréquences relatives d'émission des relevés par équipement (inférieur à 1mn)",
                  xaxis_title="Différence moyenne entre deux relevés", 
                  yaxis_title="Nombre d'équipements",
                  font_family="Arial",
                font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1100, height=400,
                template="simple_white")
fig.show()

In [ ]:
df[(df.timedelta < pd.Timedelta(60, unit='s'))].timedelta.mean()
df[(df.timedelta < pd.Timedelta(60, unit='s'))].timedelta.describe()

Parmi les équipements dont la fréquence moyenne de relevé est inférieure à 1mn: on compte ***517*** instances.
En moyenne, la fréquence des relevés est de ***14s***.

#### Historique inexistant
*Pour certains équipements ***(269)*** seules des factures à date identiques ont été retrouvées*.  
Sachant que chaque équipement est associé à une date unique, l'analyse suivante montre la répartition de ces équipements sur le temps.

In [ ]:
no_freq = df[df.avgDiff==0]
device_ids = no_freq['device_id'].tolist()

result = client.query_df(f"""SELECT device_id, payload_timestamp, payload_source, payload_value, ingested_at, payload_metric_id \
FROM {table} \
WHERE device_id IN {device_ids} \
settings max_rows_to_read = {maxi}""")

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Histogram(x = result.payload_timestamp,
        marker={'color': green}
    )
)
fig.update_yaxes(type="log")
fig.update_layout(title=f"Distribution sur le temps des équipements ayant des factures aux dates identiques.",
                  xaxis_title="Date des relevés associés", 
                  yaxis_title="Nombre d'équipement <br> (échelle logarithmique)",
                  font_family="Arial",
                font_color=black,
                title_font_family="Times New Roman",
                title_font_color=black,
                width=1100, height=400,
                template="simple_white")
fig.show()